# Full-device EPLG

In [1]:
# (C) Copyright IBM 2026.
#
# This code is licensed under the Apache License, Version 2.0. You may
# obtain a copy of this license in the LICENSE.txt file in the root directory
# of this source tree or at http://www.apache.org/licenses/LICENSE-2.0.
#
# Any modifications or derivative works of this code must retain this
# copyright notice, and modified files need to carry a notice indicating
# that they have been altered from the originals.

In [1]:
import json
import numpy as np
import datetime

from qiskit import *
from qiskit.quantum_info import hellinger_fidelity

from qiskit_ibm_runtime import QiskitRuntimeService, Sampler
from qiskit_experiments.library.randomized_benchmarking import LayerFidelity

import qiskit_device_benchmarking.utilities.graph_utils as gu
import qiskit_device_benchmarking.utilities.layer_fidelity_utils as lfu
import qiskit_device_benchmarking.utilities.file_utils as fu

In [2]:
import json
from datetime import datetime

def default(obj):
    if isinstance(obj, datetime):
        return {
            "__type__": "datetime",
            "value": obj.isoformat(),
        }
    raise TypeError(f"Cannot serialize {type(obj)}")

def object_hook(obj):
    if obj.get("__type__") == "datetime":
        return datetime.fromisoformat(obj["value"])
    return obj

data = {
    "name": "test",
    "created": datetime.now(),
}

In [3]:
service = QiskitRuntimeService()

In [4]:
backend = service.backend("ibm_phoenix")
coupling_map = backend.coupling_map
twoq_gate = "cz"

In [5]:
# Get one qubit basis gates
oneq_gates = []
for i in backend.configuration().basis_gates:
    # put in a case to handle rx and rzz
    if i.casefold() == "rx" or i.casefold() == "rzz":
        continue
    if i.casefold() != twoq_gate.casefold():
        oneq_gates.append(i)

In [6]:
grid_chains = lfu.get_grids(backend)

In [7]:
# there are two sets of chains that can be run in four disjoint experiments
layers = [[] for i in range(4)]
grid_chain_flt = [[], []]
for i in range(2):
    all_pairs = gu.path_to_edges(grid_chains[i], coupling_map)
    for j, pair_lst in enumerate(all_pairs):
        grid_chain_flt[i] += grid_chains[i][j]
        sub_pairs = [tuple(pair) for pair in pair_lst]  # make this is a list of tuples
        layers[2 * i] += sub_pairs[0::2]
        layers[2 * i + 1] += sub_pairs[1::2]

In [13]:
[len(ll) for ll in layers]

[60, 48, 60, 50]

In [9]:
# Check that each list is in the coupling map and is disjoint
for layer in layers:
    for qpair in layer:
        if tuple(qpair) not in coupling_map:
            raise ValueError(f"Gate on {qpair} does not exist")
        for k in layer:
            if k == qpair:
                continue
            if k[0] in qpair or k[1] in qpair:
                print(f"Warning: sets are not disjoint for gate {k} and {qpair}")

In [ ]:
# generate two experiments
lfexps = []
for i in range(2):
    lfexps.append(
        LayerFidelity(
            physical_qubits=grid_chain_flt[i],
            two_qubit_layers=layers[2 * i : (2 * i + 2)],
            lengths=[1, 2, 12, 41, 96, 187, 322, 512],
            backend=backend,
            num_samples=6,
            seed=12345,
            two_qubit_gate=twoq_gate,
            one_qubit_basis_gates=oneq_gates,
        )
    )

    # set maximum number of circuits per job to avoid errors due to too large payload
    lfexps[i].experiment_options.max_circuits = 144

In [ ]:
nshots = 1000
sampler = Sampler(backend)
sampler.options.default_shots = nshots

In [ ]:
# Run the LF experiment (generate circuits and submit the job)
exp_data_lst = []
for i in range(2):
    exp_data_lst.append(lfexps[i].run(shots=nshots, sampler=sampler))
    print(
        f"Run experiment: ID={exp_data_lst[i].experiment_id} with jobs {exp_data_lst[i].job_ids}]"
    )

In [14]:
jobs = []
jobs.extend(exp_data_lst[0].jobs())
jobs.extend(exp_data_lst[1].jobs())

In [15]:
# make an updated error map
updated_err_dicts = []
err_dict = lfu.make_error_dict(backend, twoq_gate)

for i in range(2):
    # get the results from the experiment
    df = exp_data_lst[i].analysis_results(dataframe=True)
    for j in range(2):
        updated_err_dicts.append(lfu.df_to_error_dict(df, layers[2 * i + j]))
err_dict = lfu.update_error_dict(err_dict, updated_err_dicts)

In [16]:
err_list = list(err_dict.values())

In [66]:
results = {'name': 'readout', 'backend': backend.name, 'datetime': str(jobs[0].creation_date),
           'backend_properties': json.dumps(backend.properties().to_dict(), default=default),
           'shots': 1_000, 'runtime': sum([jj.usage() for jj in jobs]),
           'lengths': [1, 2, 12, 41, 96, 187, 322, 512],
           'num_samples': 6,
           'seed': 12345,
           'layers': layers,
           'gate_errors': err_dict}

In [68]:
with open("eplg_results.json", "w") as f:
        json.dump(results, f)